# Portfolio valuation

This notebook puts three trades from the shared commodity book into one `Portfolio` and checks that the portfolio mark-to-market is the sum of the legs.

The book is:

- a purchased put and a sold call with the same maturity and settlement (one collar)
- a purchased physically settled forward, struck below the curve
- a purchased Asian forward on the USDZAR business days in June 2027, also struck below its average

`sjd.examples.sample_market` and `sjd.examples.sample_contracts` build that book so the later notebooks stay on the same trades. Open those functions to see the curve pillars and the contract constructors.

Build the module first (`cmake --build --preset debug --target sjd_python`, which copies `_core.pyd` into `python/sjd`), install once with `python -m pip install -e python`, and shut the kernel down before rebuilding.


In [1]:
import math

import sjd
from sjd import examples

print(sjd.__file__)


E:\Code\sjd\python\sjd\__init__.py


## Build the book

The forward and the Asian are struck below the market, so the book is an asset. Later notebooks use that positive mark-to-market when they post variation margin.


In [2]:
market = examples.sample_market()
book = examples.sample_contracts(market["forward"], market)
environment = market["environment"]
print("anchor", market["anchor"])
print(
    "collar", market["collar_settlement"],
    "put", book["put_strike"],
    "call", book["call_strike"],
)
print("forward", market["forward_settlement"], "strike", book["forward_strike"])
print(
    "asian", market["asian_settlement"],
    "averaging days", len(market["averaging_dates"]),
    "strike", book["asian_strike"],
)


anchor 2026-09-16
collar 2026-12-17 put 17.1 call 18.16
forward 2027-03-16 strike 17.41
asian 2027-07-02 averaging days 20 strike 17.72


## Leg values and the portfolio

`validate_report` has to succeed before `get_value`. The portfolio does not price a different product: it adds the single-instrument values.


In [3]:
portfolio = sjd.Portfolio(book["contracts"])
report = portfolio.validate_report(environment)
assert report.ok, [issue.message for issue in report.issues]

labels = ["Purchased put", "Sold call", "Forward", "Asian forward"]
leg_values = []
print(f"{'Leg':<16} {'Value':>16}")
for label, contract in zip(labels, book["contracts"]):
    value = contract.get_value(environment)
    leg_values.append(value)
    print(f"{label:<16} {value:16,.2f}")

portfolio_value = portfolio.get_value(environment)
print(f"{'Portfolio':<16} {portfolio_value:16,.2f}")
assert math.isclose(portfolio_value, sum(leg_values), rel_tol=1e-12)
assert portfolio_value > 0.0
print("Portfolio mark-to-market matches the sum of the legs.")


Leg                         Value
Purchased put          194,413.40
Sold call             -206,937.19
Forward                340,269.68
Asian forward           83,195.69
Portfolio              410,941.59
Portfolio mark-to-market matches the sum of the legs.
